# GRPO Fine-Tuning for CSE 151B Math Reasoning

This notebook fine-tunes `Qwen/Qwen3-4B-Thinking-2507` with TRL's `GRPOTrainer`. It mirrors the starter evaluation path: MCQ rewards use boxed-letter exact match, and free-form rewards use the local `Judger.auto_judge()` symbolic/numeric evaluator.

## 1. Environment

Install these packages in the same environment/kernel used for training. The first cell is intentionally commented so rerunning the notebook does not reinstall packages by accident.

In [1]:
# # Uncomment for a fresh environment, then restart the kernel.
# !python -m pip install -U \
#     "trl>=0.17.0" \
#     "transformers>=4.51.0" \
#     accelerate datasets peft bitsandbytes safetensors \
#     sympy numpy tqdm antlr4-python3-runtime==4.11.1 ipykernel jupyter

In [2]:
import importlib, sys

packages = [
    "trl",
    "transformers",
    "accelerate",
    "datasets",
    "peft",
    "bitsandbytes",
    "safetensors",
    "sympy",
    "numpy",
    "tqdm",
    "antlr4-python3-runtime",
    "ipykernel",
    "jupyter"
]

print(f"Python: {sys.version}\n" + "-" * 50)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<20} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<20} NOT INSTALLED")

# Also check CUDA
try:
    import torch
    print(f"\n{'CUDA available':<20} {torch.cuda.is_available()}")
    print(f"{'CUDA version':<20} {torch.version.cuda}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}  |  {props.total_memory / 1e9:.1f} GB  |  SM {props.major}.{props.minor}")
except ImportError:
    pass

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
--------------------------------------------------
trl                  1.5.1
transformers         5.9.0
accelerate           1.13.0
datasets             4.8.5
peft                 0.19.1
bitsandbytes         0.49.2
safetensors          0.7.0
sympy                1.14.0
numpy                2.4.6
tqdm                 4.67.3
antlr4-python3-runtime NOT INSTALLED
ipykernel            7.2.0
jupyter              unknown

CUDA available       True
CUDA version         12.8
  GPU 0: Tesla T4  |  15.6 GB  |  SM 7.5


## 2. Imports and Configuration

**IMPORTANT:**
- Assumes working directory is /content
- Must clone 151B_SP26_Competition repo into /content

In [3]:
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Optional

import torch
from datasets import Dataset
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer


# competition_dir = Path("151B_SP26_Competition").resolve()
PROJECT_ROOT = Path("151B_SP26_Competition")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))
from judger import Judger

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = PROJECT_ROOT / "data" / "public.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "grpo-qwen3-4b-thinking"
RESULTS_PATH = PROJECT_ROOT / "results" / "grpo_eval_samples.jsonl"

SEED = 151
GPU_ID = "0"
USE_QLORA = True
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
USE_BF16 = torch.cuda.is_available() and CUDA_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

# Keep the default path cheap and safe. Set these to False/True for a real run.
DEBUG_SUBSET = True
DEBUG_TRAIN_SIZE = 32
DEBUG_EVAL_SIZE = 16
RUN_TRAINING = False                   # <-- Set to True to run the training loop
RUN_POST_TRAIN_EVAL = False            # <-- Set to True to run evaluation after training (can be done separately)

TRAIN_TEST_SPLIT = 0.08
MAX_PROMPT_LENGTH = 2048
MAX_COMPLETION_LENGTH = 1024
NUM_GENERATIONS = 4

random.seed(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"bf16: {USE_BF16}, fp16: {USE_FP16}")


CUDA available: True
bf16: False, fp16: True


## 3. Prompt Construction

The prompts intentionally match the starter notebook's format so the trained policy is rewarded for behavior that the competition scorer can extract.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list[str]]) -> list[dict[str, str]]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{label}. {option.strip()}" for label, option in zip(labels, options))
        user_prompt = f"{question}\n\nOptions:\n{opts_text}"
        return [
            {"role": "system", "content": SYSTEM_PROMPT_MCQ},
            {"role": "user", "content": user_prompt},
        ]

    return [
        {"role": "system", "content": SYSTEM_PROMPT_MATH},
        {"role": "user", "content": question},
    ]

## 4. Dataset Loading and Split

In [5]:
def load_public_dataset(path: Path) -> Dataset:
    rows = []
    with path.open() as f:
        for line in f:
            item = json.loads(line)
            options = item.get("options") or []
            answer = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
            rows.append(
                {
                    "id": item["id"],
                    "prompt": build_prompt(item["question"], options),
                    "question": item["question"],
                    "options": options,
                    "answer": answer,
                    "is_mcq": bool(options),
                }
            )
    return Dataset.from_list(rows)


full_dataset = load_public_dataset(DATA_PATH).shuffle(seed=SEED)
split = full_dataset.train_test_split(test_size=TRAIN_TEST_SPLIT, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

if DEBUG_SUBSET:
    train_dataset = train_dataset.select(range(min(DEBUG_TRAIN_SIZE, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(DEBUG_EVAL_SIZE, len(eval_dataset))))

print(train_dataset)
print(eval_dataset)
print(train_dataset[0]["prompt"][-1]["content"][:500])

Dataset({
    features: ['id', 'prompt', 'question', 'options', 'answer', 'is_mcq'],
    num_rows: 32
})
Dataset({
    features: ['id', 'prompt', 'question', 'options', 'answer', 'is_mcq'],
    num_rows: 16
})
Kelvin the Frog generates an infinite sequence $a_n$ , which satisfies the relation
$$$a_{m-n} + a_{m+n} = 2a_ma_n$$$
for any nonnegative integers $m\ge n$ . If $a_0 = 1$ , let $x$ be the largest possible value of $a_1$ such that $\prod^{\lfloor log_2{2014} \rfloor}_{i=0} a_{2^i} = \frac{1}{2048}$ . When $a_1 = x$ , compute the remainder when $|2049a_{2049}|$ is divided by $1000$ .

Options:
A. 52
B. 55
C. 54
D. 49
E. 50
F. 57
G. 53
H. 51
I. 56
J. 58


## 5. Reward Functions

`GRPOTrainer` calls custom rewards with `prompts`, `completions`, and every non-`prompt` dataset column. These functions accept `**kwargs` for compatibility with TRL's reward API.

In [6]:
judger = Judger(strict_extract=False)


def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return completion[0].get("content", "")
    return str(completion)


def extract_boxed_text(text: str) -> str:
    try:
        return judger.extract_boxed_answer(text).strip()
    except Exception:
        return ""


def extract_letter(text: str) -> str:
    boxed = extract_boxed_text(text)
    match = re.search(r"\b([A-Za-z])\b", boxed)
    if match:
        return match.group(1).upper()

    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def correctness_reward(completions, answer, is_mcq, options=None, **kwargs):
    rewards = []
    for completion, gold, mcq in zip(completions, answer, is_mcq):
        text = completion_to_text(completion)
        if mcq:
            gold_letter = gold[0] if isinstance(gold, list) else gold
            rewards.append(1.0 if extract_letter(text) == str(gold_letter).strip().upper() else 0.0)
            continue

        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=text,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards


def boxed_format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        boxed = extract_boxed_text(text)
        rewards.append(1.0 if boxed else 0.0)
    return rewards


def reasoning_structure_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        has_final = bool(re.search(r"final answer|therefore|\\boxed", text, flags=re.IGNORECASE))
        has_work = len(text.split()) >= 20
        rewards.append(1.0 if has_final and has_work else 0.0)
    return rewards


# Quick reward sanity checks against the same evaluator logic used by the starter notebook.
sample_rewards = correctness_reward(
    completions=[r"The final answer is \\boxed{105950}.", r"I choose \\boxed{F}.", r"\\boxed{A}"],
    answer=[["325*(1+325)"], ["F"], ["C"]],
    is_mcq=[False, True, True],
)
print(sample_rewards)

[1.0, 1.0, 0.0]


## 6. Tokenizer and Training Configuration

In [ ]:
import inspect

# TRL's GRPOConfig has changed across releases. Build the kwargs explicitly,
# then drop fields that the installed version does not support.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_init_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.bfloat16 if USE_BF16 else torch.float16,
    "attn_implementation": "sdpa",
}

if USE_QLORA:
    model_init_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    )

peft_config = None
if USE_QLORA:
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

grpo_kwargs = {
    # TRL requires global train/eval batch sizes to be divisible by num_generations.
    "output_dir": str(OUTPUT_DIR),
    "run_name": "qwen3-4b-thinking-grpo-cse151b",
    "seed": SEED,
    "data_seed": SEED,
    "bf16": USE_BF16,
    "fp16": USE_FP16,
    "gradient_checkpointing": True,
    "per_device_train_batch_size": NUM_GENERATIONS,
    "per_device_eval_batch_size": NUM_GENERATIONS,
    "gradient_accumulation_steps": 2 if DEBUG_SUBSET else 8,
    "learning_rate": 5e-6,
    "max_steps": 20 if DEBUG_SUBSET else 300,
    "warmup_ratio": 0.03,
    "logging_steps": 1 if DEBUG_SUBSET else 10,
    "save_steps": 10 if DEBUG_SUBSET else 50,
    "save_total_limit": 2,
    "eval_strategy": "steps",
    "eval_steps": 10 if DEBUG_SUBSET else 50,
    "report_to": "none",
    "remove_unused_columns": False,
    # Present in TRL 0.x, removed in TRL 1.5.x. Filtered below when unsupported.
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_completion_length": MAX_COMPLETION_LENGTH,
    "num_generations": NUM_GENERATIONS,
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "beta": 0.04,
    "reward_weights": [1.0, 0.2, 0.1],
    "log_completions": True,
    "model_init_kwargs": model_init_kwargs,
}

supported_grpo_args = set(inspect.signature(GRPOConfig).parameters)
unsupported_grpo_args = sorted(set(grpo_kwargs) - supported_grpo_args)
if unsupported_grpo_args:
    print(f"Dropping unsupported GRPOConfig args for this TRL version: {unsupported_grpo_args}")

grpo_kwargs = {key: value for key, value in grpo_kwargs.items() if key in supported_grpo_args}
training_args = GRPOConfig(**grpo_kwargs)

training_args




## 7. Trainer Initialization

This cell loads the model. In debug mode, it is still the expensive step; skip it if you only want to inspect data and reward functions.

In [ ]:
trainer = GRPOTrainer(
    model=MODEL_ID,
    args=training_args,
    reward_funcs=[correctness_reward, boxed_format_reward, reasoning_structure_reward],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Trainer initialized.")

## 8. Training

Set `RUN_TRAINING = True` in the configuration cell when ready. The default deliberately avoids launching a full training job.

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "final"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "final"))
    train_result
else:
    print("RUN_TRAINING is False; skipping training.")

## 9. Lightweight Post-Training Evaluation

This samples a few eval prompts, generates responses with the trainer model, and scores them through the same reward/evaluator path. It is gated separately from training.

In [ ]:
def score_response(response: str, item: dict[str, Any]) -> bool:
    if item["is_mcq"]:
        gold_letter = item["answer"][0] if isinstance(item["answer"], list) else item["answer"]
        return extract_letter(response) == str(gold_letter).strip().upper()

    gold_list = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
    except Exception:
        return False


if RUN_POST_TRAIN_EVAL:
    model = trainer.model
    model.eval()
    records = []
    for item in tqdm(eval_dataset.select(range(min(8, len(eval_dataset))))):
        prompt_text = tokenizer.apply_chat_template(
            item["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_COMPLETION_LENGTH,
                do_sample=True,
                temperature=0.6,
                top_p=0.95,
            )
        response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        records.append(
            {
                "id": item["id"],
                "is_mcq": item["is_mcq"],
                "gold": item["answer"],
                "response": response,
                "correct": score_response(response, item),
            }
        )

    with RESULTS_PATH.open("w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

    print(f"Saved {len(records)} sampled eval records to {RESULTS_PATH}")
    print(f"Sample accuracy: {sum(r['correct'] for r in records)} / {len(records)}")
else:
    print("RUN_POST_TRAIN_EVAL is False; skipping generation eval.")

## Notes and Assumptions

- The training dataset is currently `data/public.jsonl`, split into train/eval subsets.
- QLoRA is enabled by default because full fine-tuning a 4B reasoning model is likely memory-heavy.
- `RUN_TRAINING` and `RUN_POST_TRAIN_EVAL` default to `False` so opening or running the notebook top-to-bottom will not accidentally launch a long job.
- Reward correctness intentionally reuses `Judger.auto_judge()` for free-form answers to match the starter evaluator as closely as possible.